## 퍼널 분석 (Funnel Analysis)
- 목적: 주문~리뷰까지 각 단계별 소요시간 측정, 병목 구간 탐지
- 핵심 질문: 판매자가 택배사에 늦게 준 것인가, 택배사가 배송을 오래 한 것인가?
- 사용 컬럼: order_purchase_timestamp · order_approved_at · order_delivered_carrier_date · order_delivered_customer_date
- 시각화: 구간별 소요시간 히스토그램 · 박스플롯

## 코호트 분석 (Cohort Analysis)
- 목적: 배송 지연 경험 고객의 재구매율 추적
- 가설: 배송 지연 경험 고객은 정상 배송 고객보다 리텐션이 낮을 것이다
- 사용 컬럼: customer_unique_id · purchase_month · is_late · review_score
- 시각화: 재구매율 히트맵 · 그룹별 리텐션 곡선

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATE_COLS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df = pd.read_csv("funnel_df.csv", parse_dates=DATE_COLS)
valid = df[df['is_valid_funnel']].copy()

print('전체:', df.shape)
print('유효 주문:', valid.shape)
print()
print(valid[['t_approve_d','t_carrier_d','t_delivery_d']].describe().round(2))

전체: (99441, 18)
유효 주문: (95088, 18)

       t_approve_d  t_carrier_d  t_delivery_d
count     95088.00     95088.00      95088.00
mean          0.40         2.85          9.36
std           0.80         3.48          8.77
min           0.00         0.00          0.00
25%           0.01         0.90          4.11
50%           0.01         1.85          7.11
75%           0.56         3.62         12.06
max          30.89       125.76        205.19


In [6]:
# 1. order_status 확인
print('is_valid_funnel vs delivered 비교')
print('is_valid_funnel True:', df['is_valid_funnel'].sum())
print('delivered 건수:', (df['order_status'] == 'delivered').sum())
print()

# 2. valid 기준으로 확정
valid = df[df['is_valid_funnel']].copy()

# 3. review_score 결측 확인
print('review_score 결측:', valid['review_score'].isnull().sum())

# 4. outlier 확인
print()
print('t_carrier_d 99퍼센타일:', valid['t_carrier_d'].quantile(0.99).round(2))
print('t_delivery_d 99퍼센타일:', valid['t_delivery_d'].quantile(0.99).round(2))

is_valid_funnel vs delivered 비교
is_valid_funnel True: 95088
delivered 건수: 96478

review_score 결측: 639

t_carrier_d 99퍼센타일: 17.14
t_delivery_d 99퍼센타일: 41.03


In [7]:
# =============================================
# EDA - 배송 지연 → 만족도 → 재구매율 연결
# =============================================

# 1. 전체 지연율
print('=== 1. 전체 지연율 ===')
print(f"전체 유효 주문: {len(valid):,}건")
print(f"지연 주문: {valid['is_delayed'].sum():,}건")
print(f"지연율: {valid['is_delayed'].mean()*100:.1f}%")

# 2. 지연 여부별 리뷰 점수 비교
print()
print('=== 2. 지연 여부별 리뷰 점수 ===')
review_by_delay = valid.groupby('is_delayed')['review_score'].agg(['mean','median','count'])
review_by_delay.index = ['정상 배송', '지연 배송']
print(review_by_delay.round(2))

=== 1. 전체 지연율 ===
전체 유효 주문: 95,088건
지연 주문: 7,793건
지연율: 8.2%

=== 2. 지연 여부별 리뷰 점수 ===
       mean  median  count
정상 배송  4.29     5.0  86821
지연 배송  2.57     2.0   7628


In [8]:
# 3. 지연 여부별 재구매율
print('=== 3. 지연 여부별 재구매율 ===')
purchase_count = valid.groupby('customer_unique_id')['order_id'].count().reset_index()
purchase_count.columns = ['customer_unique_id', 'order_count']

first_order = valid.sort_values('order_purchase_timestamp').drop_duplicates('customer_unique_id')[['customer_unique_id','is_delayed']]
first_order = first_order.merge(purchase_count, on='customer_unique_id')
first_order['is_repurchase'] = (first_order['order_count'] >= 2).astype(int)

repurchase = first_order.groupby('is_delayed')['is_repurchase'].mean() * 100
repurchase.index = ['정상 배송', '지연 배송']
print(repurchase.round(2))

=== 3. 지연 여부별 재구매율 ===
정상 배송    3.03
지연 배송    2.47
Name: is_repurchase, dtype: float64


In [9]:
print('=== 전체 재구매율 ===')
print(f"전체 고객수: {len(purchase_count):,}명")
print(f"재구매 고객수: {(purchase_count['order_count'] >= 2).sum():,}명")
print(f"전체 재구매율: {(purchase_count['order_count'] >= 2).mean()*100:.2f}%")

=== 전체 재구매율 ===
전체 고객수: 92,031명
재구매 고객수: 2,744명
전체 재구매율: 2.98%


In [10]:
# =============================================
# EDA - 배송 소요일 vs 리뷰 점수 상관관계
# =============================================

# 1. 배송 소요일 구간별 리뷰 점수
print('=== 배송 소요일 구간별 리뷰 점수 ===')
valid['delivery_bucket'] = pd.cut(
    valid['t_total_d'],
    bins=[0, 5, 10, 15, 25, 999],
    labels=['5일 이하', '6~10일', '11~15일', '16~25일', '25일 초과']
)

bucket_review = valid.groupby('delivery_bucket')['review_score'].agg(['mean','count'])
print(bucket_review.round(2))

print()

# 2. 배송 소요일 vs 재구매율
print('=== 배송 소요일 구간별 재구매율 ===')
valid_with_repurchase = valid.merge(
    first_order[['customer_unique_id','is_repurchase']], 
    on='customer_unique_id', how='left'
)

bucket_repurchase = valid_with_repurchase.groupby('delivery_bucket')['is_repurchase'].mean() * 100
print(bucket_repurchase.round(2))

=== 배송 소요일 구간별 리뷰 점수 ===
                 mean  count
delivery_bucket             
5일 이하            4.45  12996
6~10일            4.36  32264
11~15일           4.27  23241
16~25일           4.04  18338
25일 초과           2.70   7610

=== 배송 소요일 구간별 재구매율 ===
delivery_bucket
5일 이하     6.02
6~10일     6.02
11~15일    6.37
16~25일    6.26
25일 초과    5.41
Name: is_repurchase, dtype: float64


In [1]:
import pandas as pd

DATE_COLS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df = pd.read_csv("funnel_df.csv", parse_dates=DATE_COLS)
valid = df[df['is_valid_funnel']].copy()

# order_items에서 shipping_limit_date 가져오기
items = pd.read_csv("../../../Funnel_Cohort_NG/data/archive (1)/olist_order_items_dataset.csv",
                    parse_dates=['shipping_limit_date'])

items_agg = items.groupby('order_id').agg(
    shipping_limit_date=('shipping_limit_date', 'min')
).reset_index()

valid = valid.merge(items_agg, on='order_id', how='left')

# 판매자 지각 여부
valid['is_seller_late'] = (
    valid['order_delivered_carrier_date'] > valid['shipping_limit_date']
).astype(float)

print('판매자 지각률:', round(valid['is_seller_late'].mean()*100, 1), '%')
print('판매자 지각 건수:', int(valid['is_seller_late'].sum()), '건')

판매자 지각률: 9.2 %
판매자 지각 건수: 8701 건


In [2]:
# 골든타임 초과 주문 중 판매자 귀책 vs 택배사 귀책
over_golden = valid[valid['t_total_d'] >= 21].copy()

print(f'골든타임(21일) 초과 주문: {len(over_golden):,}건')
print(f'그 중 판매자 지각: {int(over_golden["is_seller_late"].sum()):,}건 ({over_golden["is_seller_late"].mean()*100:.1f}%)')
print(f'그 중 택배사 귀책: {int((over_golden["is_seller_late"]==0).sum()):,}건 ({(over_golden["is_seller_late"]==0).mean()*100:.1f}%)')

골든타임(21일) 초과 주문: 12,450건
그 중 판매자 지각: 2,919건 (23.4%)
그 중 택배사 귀책: 9,531건 (76.6%)


In [3]:
from scipy import stats

group_before = valid[valid['t_total_d'] < 21]['review_score'].dropna()
group_after = valid[valid['t_total_d'] >= 21]['review_score'].dropna()

stat, p = stats.mannwhitneyu(group_before, group_after, alternative='greater')

print(f'21일 이전 그룹: {len(group_before):,}건 / 평균 리뷰 {group_before.mean():.2f}점')
print(f'21일 이후 그룹: {len(group_after):,}건 / 평균 리뷰 {group_after.mean():.2f}점')
print()
print(f'Mann-Whitney U 통계량: {stat:.2f}')
print(f'p-value: {p:.10f}')
print('결론:', '통계적으로 유의함 (p < 0.05)' if p < 0.05 else '유의하지 않음')

21일 이전 그룹: 82,186건 / 평균 리뷰 4.31점
21일 이후 그룹: 12,263건 / 평균 리뷰 3.12점

Mann-Whitney U 통계량: 708091975.00
p-value: 0.0000000000
결론: 통계적으로 유의함 (p < 0.05)


In [5]:
# =============================================
# 느린 지역 Top5 수치 정리
# 목적: 골든타임(21일) 초과율이 높은 지역 파악
#       → B팀(승근님/호영님) 허브 위치 선정 근거 자료
# 출처: 다연님 EDA에서 느린 Top5 지역 확인 (PA, MA, CE, PB, BA)
# =============================================
slow_states = ['PA', 'MA', 'CE', 'PB', 'BA']
top5_slow = valid[valid['customer_state'].isin(slow_states)]
summary = top5_slow.groupby('customer_state').agg(
    평균배송일=('t_total_d', 'mean'),
    중앙값배송일=('t_total_d', 'median'),
    골든타임초과율=('t_total_d', lambda x: (x >= 21).mean() * 100),
    평균리뷰=('review_score', 'mean'),
    주문수=('t_total_d', 'count')
).round(2)

print('=== 느린 지역 Top5 배송 현황 ===')
print(summary)
print()
print('* 골든타임 기준: 21일 (Mann-Whitney 통계 검정 완료)')
print('* 골든타임 초과율이 높을수록 B팀 허브 우선 설치 대상')

=== 느린 지역 Top5 배송 현황 ===
                평균배송일  중앙값배송일  골든타임초과율  평균리뷰   주문수
customer_state                                    
BA              19.38   16.95    32.59  3.93  3210
CE              21.40   18.23    38.20  3.94  1259
MA              21.68   19.24    42.05  3.84   704
PA              23.81   21.09    50.75  3.90   936
PB              20.51   18.20    38.31  4.07   509

* 골든타임 기준: 21일 (Mann-Whitney 통계 검정 완료)
* 골든타임 초과율이 높을수록 B팀 허브 우선 설치 대상


In [6]:
# =============================================
# 골든타임 찾기 - 기울기 변화 최대인 날
# 목적: 주관적 기준(4점, 2점)이 아닌
#       데이터 기반으로 만족도가 가장 급격히 꺾이는 날 탐지
# =============================================

import numpy as np

# 하루씩 리뷰 평균 계산 (30건 이상인 날만)
daily = valid.groupby(valid['t_total_d'].astype(int))['review_score'].agg(['mean','count'])
daily.columns = ['avg_review', 'n']
daily = daily[daily['n'] >= 30]

# 하루씩 기울기 계산 (당일 - 전날)
daily['slope'] = daily['avg_review'].diff()

# 기울기가 가장 급격히 떨어지는 날
golden_day = daily['slope'].idxmin()

print('=== 기울기 변화 기준 골든타임 ===')
print(f'만족도가 가장 급격히 꺾이는 날: {golden_day}일')
print(f'해당 날 평균 리뷰: {daily.loc[golden_day, "avg_review"]:.2f}점')
print(f'기울기(전날 대비 하락): {daily.loc[golden_day, "slope"]:.4f}점')
print()
print('상위 5개 급락 구간:')
print(daily.nsmallest(5, 'slope')[['avg_review','slope','n']])

=== 기울기 변화 기준 골든타임 ===
만족도가 가장 급격히 꺾이는 날: 41일
해당 날 평균 리뷰: 1.83점
기울기(전날 대비 하락): -0.3830점

상위 5개 급락 구간:
           avg_review     slope    n
t_total_d                           
41           1.834395 -0.382996  157
44           1.707547 -0.301381  106
28           3.086957 -0.262889  575
31           2.761773 -0.228815  361
37           2.078125 -0.215208  192


In [7]:
# =============================================
# 골든타임 찾기 - 3.5점 아래로 처음 떨어지는 날
# 표본 충분한 날(100건 이상)만 대상
# =============================================

daily_filtered = daily[daily['n'] >= 100]

# 3.5점 아래로 처음 떨어지는 날
below_35 = daily_filtered[daily_filtered['avg_review'] < 3.5]

if len(below_35) > 0:
    golden_day2 = below_35.index.min()
    print(f'3.5점 아래로 처음 떨어지는 날: {golden_day2}일')
    print(f'해당 날 평균 리뷰: {daily_filtered.loc[golden_day2, "avg_review"]:.2f}점')
    print()

# 25일~30일 구간 상세히 보기
print('=== 25~35일 구간 상세 ===')
print(daily_filtered.loc[25:35, ['avg_review','slope','n']].round(3))

3.5점 아래로 처음 떨어지는 날: 25일
해당 날 평균 리뷰: 3.47점

=== 25~35일 구간 상세 ===
           avg_review  slope    n
t_total_d                        
25              3.466 -0.163  804
26              3.494  0.028  698
27              3.350 -0.144  646
28              3.087 -0.263  575
29              3.081 -0.006  468
30              2.991 -0.091  425
31              2.762 -0.229  361
32              2.547 -0.215  329
33              2.616  0.069  310
34              2.444 -0.172  286
35              2.348 -0.096  276


In [8]:
# =============================================
# 골든타임 통계 검정 - 나경님 기준 25일
# 목적: 25일 임계점이 통계적으로 유의미한지 검증
# 방법: Mann-Whitney U 검정
# =============================================

from scipy import stats

group_before = valid[valid['t_total_d'] < 25]['review_score'].dropna()
group_after = valid[valid['t_total_d'] >= 25]['review_score'].dropna()

stat, p = stats.mannwhitneyu(group_before, group_after, alternative='greater')

print('=== 골든타임 25일 통계 검정 ===')
print(f'25일 이전 그룹: {len(group_before):,}건 / 평균 리뷰 {group_before.mean():.2f}점')
print(f'25일 이후 그룹: {len(group_after):,}건 / 평균 리뷰 {group_after.mean():.2f}점')
print()
print(f'Mann-Whitney U 통계량: {stat:.2f}')
print(f'p-value: {p:.10f}')
print('결론:', '통계적으로 유의함 (p < 0.05)' if p < 0.05 else '유의하지 않음')

=== 골든타임 25일 통계 검정 ===
25일 이전 그룹: 86,839건 / 평균 리뷰 4.28점
25일 이후 그룹: 7,610건 / 평균 리뷰 2.70점

Mann-Whitney U 통계량: 500736013.00
p-value: 0.0000000000
결론: 통계적으로 유의함 (p < 0.05)
